# 🔬 MetaboNet — Lazy EDA (RAM-Safe for 132M Rows)

**Problem:** The full dataset is ~132M rows → OOM if loaded via `read_parquet`.  
**Solution:** Use `polars.scan_parquet()` (LazyFrame) for all aggregations, then deep-dive on a single "Golden Patient".  

---

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────
!pip install polars plotly pyarrow -q

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import os
import gc

# ── Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

print(f"Polars version: {pl.__version__}")
print(f"Run at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ── File Paths ────────────────────────────────────────────────────────
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

TRAIN_PATH = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
TEST_PATH  = f"{BASE_PATH}/raw/metabonet_public_test.parquet"

print(f"📁 Train: {TRAIN_PATH} — exists: {os.path.exists(TRAIN_PATH)}")
print(f"📁 Test : {TEST_PATH}  — exists: {os.path.exists(TEST_PATH)}")

---
## STEP 1 — Sanity Check (Lazy Mode) 🔍
Fetch only 5 rows to confirm the schema is clean.
**No full load — zero RAM risk.**

In [ ]:
# ── 1.1 Lazy Scan ────────────────────────────────────────────────────
q = pl.scan_parquet(TRAIN_PATH)

# Print schema without loading data
schema = q.collect_schema()
print("SCHEMA (from lazy scan — 0 bytes loaded):")
for col_name in schema.names():
    print(f"  {col_name:30s} → {schema[col_name]}")
print(f"\nTotal columns: {len(schema.names())}")

In [ ]:
# ── 1.2 Fetch 5 rows to prove columns are readable ───────────────────
sample = q.head(5).collect()

# Show key columns
key_cols = [c for c in ["id", "gender", "cgm_device", "date", "CGM"] if c in sample.columns]
print("SAMPLE (5 rows, key columns):")
print(sample.select(key_cols))

# Prove dtypes
print("\nDATATYPE PROOF:")
for c in key_cols:
    val = sample[c][0]
    print(f"  {c:20s} dtype={sample[c].dtype}  sample_value={val!r}")

---
## STEP 2 — Global Aggregations (RAM Efficient) 📊
All queries run lazily — only the tiny result is collected.

In [ ]:
# ── 2.1 Total rows & unique patients ─────────────────────────────────
global_stats = (
    q.select(
        pl.col("id").n_unique().alias("n_patients"),
        pl.len().alias("total_rows"),
    )
    .collect()
)

n_patients = global_stats["n_patients"][0]
total_rows = global_stats["total_rows"][0]

print("╔══════════════════════════════════════════╗")
print("║        📋  GLOBAL DATASET STATS          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Total rows     : {total_rows:>18,}   ║")
print(f"║  Unique patients: {n_patients:>18,}   ║")
print(f"║  Avg rows/patient: {total_rows // n_patients:>17,}   ║")
print("╚══════════════════════════════════════════╝")

In [ ]:
# ── 2.2 Top patients by row count ────────────────────────────────────
top_patients = (
    q.group_by("id")
    .agg(pl.len().alias("row_count"))
    .sort("row_count", descending=True)
    .limit(20)
    .collect()
)

print("TOP 20 PATIENTS (by data volume):")
print(top_patients)

# Identify the Golden Patient
GOLDEN_ID = top_patients["id"][0]
GOLDEN_ROWS = top_patients["row_count"][0]
print(f"\n⭐ Golden Patient: '{GOLDEN_ID}' with {GOLDEN_ROWS:,} rows")

In [ ]:
# ── 2.3 Bar chart: Top 20 patients ───────────────────────────────────
fig = px.bar(
    top_patients.to_pandas(),
    x="id", y="row_count",
    title="📊 Top 20 Patients by Data Volume",
    labels={"id": "Patient ID", "row_count": "Rows"},
    color="row_count",
    color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_dark", height=450, xaxis_tickangle=-45)
fig.show()

In [ ]:
# ── 2.4 Global demographics (lazy) ───────────────────────────────────
for demo_col in ["gender", "ethnicity", "cgm_device", "insulin_delivery_device"]:
    if demo_col in schema.names():
        counts = (
            q.group_by(demo_col)
            .agg(pl.len().alias("count"))
            .sort("count", descending=True)
            .limit(10)
            .collect()
        )
        print(f"\n{'─'*50}")
        print(f"Column: {demo_col}")
        print(counts)

In [ ]:
# ── 2.5 Global null analysis (lazy) ──────────────────────────────────
null_exprs = [
    pl.col(c).null_count().alias(f"{c}__nulls")
    for c in schema.names()
]
null_counts = q.select(null_exprs + [pl.len().alias("__total")]).collect()

total = null_counts["__total"][0]

print("╔══════════════════════════════════════════════════════════════╗")
print("║             📋  GLOBAL NULL REPORT (LAZY)                  ║")
print("╠════════════════════════════╦═══════════╦═══════════════════╣")
print("║ Column                     ║ Null Count║ Null %            ║")
print("╠════════════════════════════╬═══════════╬═══════════════════╣")

null_report = []
for c in schema.names():
    n = null_counts[f"{c}__nulls"][0]
    p = (n / total) * 100
    null_report.append((c, n, p))
    print(f"║ {c:26s} ║ {n:>9,} ║ {p:>15.2f}%  ║")

print("╚════════════════════════════╩═══════════╩═══════════════════╝")

mostly_null = [c for c, _, p in null_report if p > 80]
sparse_cols = [c for c, _, p in null_report if 30 < p <= 80]
usable_cols = [c for c, _, p in null_report if p <= 30]

print(f"\n🔴 Mostly empty (>80%): {mostly_null}")
print(f"🟡 Sparse (30–80%):    {sparse_cols}")
print(f"🟢 Usable (≤30%):      {usable_cols}")

In [ ]:
# ── 2.6 Null heatmap ─────────────────────────────────────────────────
null_df = pl.DataFrame({
    "column": [c for c, _, _ in null_report],
    "null_pct": [p for _, _, p in null_report],
}).sort("null_pct", descending=True).to_pandas()

fig = px.bar(
    null_df, x="column", y="null_pct",
    title="🗑️ Null Percentage per Column (Global, Lazy)",
    labels={"column": "Column", "null_pct": "Null %"},
    color="null_pct",
    color_continuous_scale="RdYlGn_r",
    range_color=[0, 100],
)
fig.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80%")
fig.add_hline(y=30, line_dash="dash", line_color="orange", annotation_text="30%")
fig.update_layout(template="plotly_dark", height=500, xaxis_tickangle=-45)
fig.show()

---
## STEP 3 — Deep Dive: The "Golden Patient" 🏆
Load only 1 patient into RAM → safe even on free Colab.

In [ ]:
# ── 3.1 Load ONLY the Golden Patient ─────────────────────────────────
df_patient = (
    q.filter(pl.col("id") == GOLDEN_ID)
    .sort("date")
    .collect()
)

print(f"⭐ Golden Patient: '{GOLDEN_ID}'")
print(f"   Rows loaded: {len(df_patient):,}")
print(f"   RAM usage:   ~{df_patient.estimated_size('mb'):.1f} MB")
print(f"   Columns:     {df_patient.columns}")
print(f"\nFirst 3 rows:")
df_patient.head(3)

In [ ]:
# ── 3.2 Temporal Parsing ─────────────────────────────────────────────
# Check if date is already datetime or needs conversion
date_dtype = df_patient["date"].dtype
print(f"Date column dtype: {date_dtype}")

if date_dtype == pl.Datetime or str(date_dtype).startswith("Datetime"):
    print("✅ Date is already Datetime. No conversion needed.")
    df_patient = df_patient.with_columns(pl.col("date").alias("datetime"))
elif date_dtype in [pl.Int64, pl.UInt64]:
    # Try nanoseconds
    sample_val = df_patient["date"][0]
    print(f"   Sample raw value: {sample_val}")
    
    df_patient = df_patient.with_columns(
        pl.from_epoch(pl.col("date"), time_unit="ns").alias("datetime")
    )
    test_year = df_patient["datetime"][0].year
    
    if test_year < 2000 or test_year > 2030:
        print(f"   ⚠️ ns → year {test_year}. Trying microseconds...")
        df_patient = df_patient.with_columns(
            pl.from_epoch(pl.col("date"), time_unit="us").alias("datetime")
        )
        test_year = df_patient["datetime"][0].year
    
    print(f"   ✅ Parsed → year {test_year}")
else:
    print(f"   ⚠️ Unexpected dtype: {date_dtype}. Using as-is.")
    df_patient = df_patient.with_columns(pl.col("date").alias("datetime"))

print(f"\nDate range: {df_patient['datetime'].min()} → {df_patient['datetime'].max()}")

In [ ]:
# ── 3.3 Gap Analysis ─────────────────────────────────────────────────
df_patient = df_patient.sort("datetime")

df_patient = df_patient.with_columns(
    (pl.col("datetime") - pl.col("datetime").shift(1)).alias("time_diff")
)
df_patient = df_patient.with_columns(
    (pl.col("time_diff").dt.total_seconds() / 60.0).alias("gap_min")
)

valid = df_patient.filter(pl.col("gap_min").is_not_null())
total_gaps = len(valid)
exact_5  = valid.filter(pl.col("gap_min") == 5.0).height
near_5   = valid.filter((pl.col("gap_min") >= 4.5) & (pl.col("gap_min") <= 5.5)).height
pct_exact = (exact_5 / total_gaps) * 100
pct_near  = (near_5 / total_gaps) * 100

print("╔══════════════════════════════════════════════════════╗")
print(f"║  ⏱️  GAP ANALYSIS — Patient '{GOLDEN_ID}'")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Total gaps analyzed : {total_gaps:>15,}            ║")
print(f"║  Exactly 5 min       : {exact_5:>15,} ({pct_exact:.1f}%)   ║")
print(f"║  Near 5 min (±30s)   : {near_5:>15,} ({pct_near:.1f}%)   ║")
print(f"║  Min gap (min)       : {valid['gap_min'].min():>15.1f}            ║")
print(f"║  Max gap (min)       : {valid['gap_min'].max():>15.1f}            ║")
print(f"║  Median gap (min)    : {valid['gap_min'].median():>15.1f}            ║")
print("╚══════════════════════════════════════════════════════╝")

In [ ]:
# ── 3.4 Session Detection ────────────────────────────────────────────
SESSION_THRESH = 15  # minutes

df_patient = df_patient.with_columns(
    ((pl.col("gap_min") > SESSION_THRESH) | pl.col("gap_min").is_null())
    .cast(pl.Int32)
    .cum_sum()
    .alias("session_id")
)

n_sessions = df_patient["session_id"].max()

session_sizes = (
    df_patient.group_by("session_id")
    .agg(pl.len().alias("size"))
    .sort("size", descending=True)
)

print(f"Sessions detected: {n_sessions}")
print(f"Avg session size:  {session_sizes['size'].mean():.0f} rows")
print(f"Largest session:   {session_sizes['size'].max()} rows")
print(f"Smallest session:  {session_sizes['size'].min()} rows")

In [ ]:
# ── 3.5 Gap Histogram (exclude standard 5-min) ───────────────────────
outliers = valid.filter(
    (pl.col("gap_min") < 4.5) | (pl.col("gap_min") > 5.5)
)

print(f"Non-standard gaps: {len(outliers):,} ({len(outliers)/total_gaps*100:.2f}%)")

fig = px.histogram(
    outliers.select(pl.col("gap_min").clip(0, 120)).to_pandas(),
    x="gap_min", nbins=80,
    title=f"🔍 Non-Standard Gaps — Patient '{GOLDEN_ID}' (clipped at 120 min)",
    labels={"gap_min": "Gap (minutes)"},
    color_discrete_sequence=["#FF6B6B"],
)
fig.update_layout(template="plotly_dark", height=400, bargap=0.05)
fig.show()

---
## STEP 4 — Target Variable Analysis (Golden Patient) 🩸

In [ ]:
# ── 4.1 CGM Stats ────────────────────────────────────────────────────
cgm_col = "CGM" if "CGM" in df_patient.columns else "cgm"

cgm_nulls = df_patient[cgm_col].null_count()
cgm_valid = df_patient.filter(pl.col(cgm_col).is_not_null())

print(f"CGM column: '{cgm_col}'")
print(f"  Null: {cgm_nulls:,} / {len(df_patient):,} ({cgm_nulls/len(df_patient)*100:.2f}%)")
print(f"  Min:  {cgm_valid[cgm_col].min()}")
print(f"  Max:  {cgm_valid[cgm_col].max()}")
print(f"  Mean: {cgm_valid[cgm_col].mean():.2f}")
print(f"  Std:  {cgm_valid[cgm_col].std():.2f}")
print(f"  Median: {cgm_valid[cgm_col].median():.2f}")

In [ ]:
# ── 4.2 CGM Distribution ─────────────────────────────────────────────
cgm_pd = cgm_valid.select(pl.col(cgm_col).cast(pl.Float64)).to_pandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=("CGM Histogram", "CGM Box Plot"))

fig.add_trace(
    go.Histogram(x=cgm_pd[cgm_col].dropna(), nbinsx=80,
                 marker_color="#4ECDC4", name="CGM"),
    row=1, col=1
)
fig.add_trace(
    go.Box(y=cgm_pd[cgm_col].dropna(), marker_color="#FF6B6B", name="CGM"),
    row=1, col=2
)

fig.update_layout(
    title=f"🩸 CGM Distribution — Patient '{GOLDEN_ID}'",
    template="plotly_dark", height=400, showlegend=False,
)
fig.show()

In [ ]:
# ── 4.3 Full CGM Trace ───────────────────────────────────────────────
trace_pd = cgm_valid.select(["datetime", cgm_col]).sort("datetime").to_pandas()

fig = px.line(
    trace_pd, x="datetime", y=cgm_col,
    title=f"🩸 Full CGM Trace — Patient '{GOLDEN_ID}' ({len(trace_pd):,} readings)",
    labels={"datetime": "Time", cgm_col: "CGM (mg/dL)"},
)
fig.update_traces(line=dict(width=0.6, color="#00D4AA"))
fig.add_hline(y=70,  line_dash="dash", line_color="red",    annotation_text="Hypo (70)")
fig.add_hline(y=180, line_dash="dash", line_color="orange", annotation_text="Hyper (180)")
fig.add_hline(y=250, line_dash="dash", line_color="red",    annotation_text="Critical (250)")
fig.update_layout(template="plotly_dark", height=500, xaxis_rangeslider_visible=True)
fig.show()

In [ ]:
# ── 4.4 Zoomed 24h Window ────────────────────────────────────────────
mid = len(trace_pd) // 2
t0 = trace_pd["datetime"].iloc[mid]
t1 = t0 + timedelta(hours=24)

zoom = trace_pd[(trace_pd["datetime"] >= t0) & (trace_pd["datetime"] <= t1)]

fig = px.line(
    zoom, x="datetime", y=cgm_col,
    title=f"🔎 24-Hour Zoom — Patient '{GOLDEN_ID}'",
    labels={"datetime": "Time", cgm_col: "CGM (mg/dL)"},
)
fig.update_traces(line=dict(width=1.5, color="#FFD93D"), mode="lines+markers")
fig.add_hline(y=70,  line_dash="dash", line_color="red")
fig.add_hline(y=180, line_dash="dash", line_color="orange")
fig.update_layout(template="plotly_dark", height=400)
fig.show()

---
## STEP 5 — The Verdict 📜

In [ ]:
# ── 5. THE VERDICT ───────────────────────────────────────────────────

# Check if key columns are Strings (not List[Int])
string_cols_ok = all(
    schema[c] in [pl.Utf8, pl.String]
    for c in ["id", "gender"] if c in schema.names()
)

freq_ok = pct_near > 90
cgm_ok = cgm_nulls / len(df_patient) < 0.1

print("")
print("╔══════════════════════════════════════════════════════════════════╗")
print("║              📋  FINAL VERDICT (LAZY EDA)                      ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                                                                ║")

s1 = "✅ YES — id/gender are native Strings" if string_cols_ok else "❌ NO — need decoding"
print(f"║  1. Columns readable?   {s1:41s} ║")

s2 = f"✅ YES — {pct_near:.1f}% within 5±0.5 min" if freq_ok else f"⚠️  NO — only {pct_near:.1f}%"
print(f"║  2. High-frequency?     {s2:41s} ║")

s3 = f"✅ OK — {cgm_nulls/len(df_patient)*100:.1f}% null" if cgm_ok else f"⚠️  HIGH — {cgm_nulls/len(df_patient)*100:.1f}% null"
print(f"║  3. CGM target quality? {s3:41s} ║")

print("║                                                                ║")
print(f"║  Dataset:  {total_rows:>12,} rows × {len(schema.names())} columns                    ║")
print(f"║  Patients: {n_patients:>12,}                                         ║")
print(f"║  Sample:   Patient '{GOLDEN_ID}' ({GOLDEN_ROWS:,} rows)")
print("║                                                                ║")

all_ok = string_cols_ok and freq_ok and cgm_ok
verdict = "✅ GO — Dataset structure valid for forecasting" if all_ok else "⚠️  NEEDS ATTENTION — See warnings above"
print(f"║  VERDICT: {verdict:53s} ║")

print("║                                                                ║")
print("║  🔑 Features to consider (≤30% null):                         ║")
for c in usable_cols:
    print(f"║     🟢 {c:56s}  ║")
print("║  ⚠️  Columns to discard (>80% null):                          ║")
for c in mostly_null:
    print(f"║     🔴 {c:56s}  ║")
print("║                                                                ║")
print("╚══════════════════════════════════════════════════════════════════╝")

# Clean up
del df_patient
gc.collect()
print("\n🧹 Patient DataFrame freed from RAM.")